In [ ]:
%pip install -q yt-dlp

In [ ]:
!node --version

In [ ]:
!sudo apt-get update && sudo apt-get install -y nodejs

In [ ]:
!node --version


In [ ]:
!sudo apt-get install -y ffmpeg

In [ ]:
from pathlib import Path
import subprocess

audio_dir = Path.home() / "music" / "audio"
audio_dir.mkdir(parents=True, exist_ok=True)
url = "https://www.youtube.com/playlist?list=PLa18oOsb45rd1xlaHXWDj0dPcbHI7EQ8s"

subprocess.run([
    "yt-dlp", "--js-runtimes", "node",
    "-x", "--audio-format", "wav",
    "-o", str(audio_dir / "%(title)s.%(ext)s"),
    url,
], check=True)

In [ ]:
!pip install librosa scikit-learn

In [ ]:
from pathlib import Path
import librosa
from IPython.display import Audio, display

file = Path.home() / "music/audio/Billie Eilish - bad guy.wav"
y, sr = librosa.load(file, duration=20.0)
display(Audio(y, rate=sr))

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Audio

file = "/home/ubuntu/music/audio/Billie Eilish - bad guy.wav"

# Load whole track (preserve stereo for playback; analyze mono)
audio, sr = librosa.load(file, sr=None, mono=False)
y = librosa.to_mono(audio) if audio.ndim == 2 else audio

duration = librosa.get_duration(y=y, sr=sr)
tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
rms = librosa.feature.rms(y=y)[0]
centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

print(f"Duration: {duration:.1f} sec | Sample rate: {sr} Hz")
print(f"Estimated tempo: {float(np.asarray(tempo).reshape(-1)[0]):.1f} BPM")
print(f"Mean loudness (RMS): {rms.mean():.4f}")
print(f"Mean spectral brightness: {centroid.mean():.0f} Hz")
print("MFCC shape:", mfcc.shape)

fig, ax = plt.subplots(3, 1, figsize=(14, 10))
librosa.display.waveshow(y, sr=sr, ax=ax[0])
ax[0].set(title="Waveform", xlabel="Time (s)")
librosa.display.specshow(
    librosa.amplitude_to_db(
        np.abs(librosa.stft(y)), ref=np.max
    ),
    sr=sr, x_axis="time", y_axis="log", ax=ax[1]
)
ax[1].set(title="Spectrogram (dB)")
librosa.display.specshow(
    librosa.power_to_db(
        librosa.feature.melspectrogram(y=y, sr=sr), ref=np.max
    ),
    sr=sr, x_axis="time", y_axis="mel", ax=ax[2]
)
ax[2].set(title="Mel spectrogram")
plt.tight_layout()
plt.show()

display(Audio(audio, rate=sr))

In [ ]:
!pip install demucs luigi

In [ ]:
# stem separation for downloaded audio

from pathlib import Path
import subprocess
import sys

audio = Path("../audio/Billie Eilish - bad guy.wav")
out = Path("../results/demucs/bad_guy")
out.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, "-m", "demucs",
    "-n", "htdemucs",
    "--two-stems=vocals",
    "-o", str(out),
    str(audio),
], check=True)

# Librosa

In [ ]:
# Basic summaries

from pathlib import Path
import numpy as np
import pandas as pd
import librosa

files = {
    "original": Path("../audio/Billie Eilish - bad guy.wav"),
    "vocals": Path("../results/demucs/bad_guy/htdemucs/Billie Eilish - bad guy/vocals.wav"),
    "no_vocals": Path("../results/demucs/bad_guy/htdemucs/Billie Eilish - bad guy/no_vocals.wav"),
}

pitch_classes = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
summary, chroma_rows = [], []

for part, path in files.items():
    if not path.exists():
        print(f"Skipping missing file: {path}")
        continue

    y, sr = librosa.load(path, sr=None, mono=True)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    tempo = librosa.feature.tempo(y=y, sr=sr)[0]

    summary.append({
        "part": part,
        "file": str(path),
        "duration_s": round(len(y) / sr, 2),
        "sample_rate": sr,
        "rms_mean": float(librosa.feature.rms(y=y).mean()),
        "tempo_bpm_estimate": float(tempo),
        "spectral_contrast_mean": float(contrast.mean()),
        **{f"mfcc_{i+1}_mean": float(mfcc[i].mean()) for i in range(13)},
    })
    chroma_rows.extend(
        {"part": part, "pitch_class": note, "mean_chroma": float(chroma[i].mean())}
        for i, note in enumerate(pitch_classes)
    )

results = Path("../results")
results.mkdir(parents=True, exist_ok=True)
pd.DataFrame(summary).to_csv(results / "audio_feature_summary.csv", index=False)
pd.DataFrame(chroma_rows).to_csv(results / "audio_chroma_profile.csv", index=False)

pd.DataFrame(summary)

In [ ]:
# Segment analysis
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt

audio = Path("../audio/Billie Eilish - bad guy.wav")
out_dir = Path("../results/librosa/segment")
out_dir.mkdir(parents=True, exist_ok=True)

y, sr = librosa.load(audio, sr=None, mono=True)
hop = 512

# Self-similarity of short-time chroma features
chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop)
recurrence = librosa.segment.recurrence_matrix(
    chroma, mode="affinity", self=True, width=3
)

# Coarse structural boundaries; number of segments is a tunable choice
boundaries = librosa.segment.agglomerative(chroma, k=8)
boundary_times = librosa.frames_to_time(boundaries, sr=sr, hop_length=hop)

pd.DataFrame({
    "boundary_frame": boundaries,
    "time_seconds": boundary_times,
}).to_csv(out_dir / "segment_boundaries.csv", index=False)

np.save(out_dir / "recurrence_matrix.npy", recurrence)

plt.figure(figsize=(8, 6))
librosa.display.specshow(
    recurrence, x_axis="time", y_axis="time",
    sr=sr, hop_length=hop, cmap="magma"
)
plt.title("Within-track chroma self-similarity")
plt.tight_layout()
plt.savefig(out_dir / "recurrence_matrix.png", dpi=160)
plt.show()

print("Segment boundaries (seconds):", np.round(boundary_times, 2))
print("Saved to:", out_dir)

In [ ]:
# Sequence analysis

from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt

reference = Path("../audio/Billie Eilish - bad guy.wav")
candidate = Path("../results/demucs/bad_guy/htdemucs/Billie Eilish - bad guy/no_vocals.wav")
out_dir = Path("../results/librosa/sequence")
out_dir.mkdir(parents=True, exist_ok=True)

def chroma_features(path):
    y, sr = librosa.load(path, sr=22050, mono=True)
    return librosa.feature.chroma_cqt(y=y, sr=sr)

X = chroma_features(reference)
Y = chroma_features(candidate)

# Normalize each frame so comparison emphasizes pitch-class pattern
X = librosa.util.normalize(X, axis=0)
Y = librosa.util.normalize(Y, axis=0)

D, wp = librosa.sequence.dtw(X=X, Y=Y, metric="cosine")
alignment_cost = float(D[-1, -1] / len(wp))

pd.DataFrame({
    "reference_frame": wp[::-1, 0],
    "candidate_frame": wp[::-1, 1],
}).to_csv(out_dir / "dtw_alignment.csv", index=False)

plt.figure(figsize=(8, 6))
librosa.display.specshow(D, x_axis="frames", y_axis="frames", cmap="magma")
plt.plot(wp[:, 1], wp[:, 0], color="cyan", linewidth=1)
plt.title(f"DTW chroma alignment (mean accumulated cost: {alignment_cost:.3f})")
plt.tight_layout()
plt.savefig(out_dir / "dtw_alignment.png", dpi=160)
plt.show()

print(f"Normalized accumulated DTW cost: {alignment_cost:.4f}")
print("Saved alignment and plot to:", out_dir)
